# 04 — Rapport par organe

Ce notebook génère un rapport HTML pour un organe spécifique, au niveau AP-HP ou d'un GHU.

**Paramètres papermill** : `ORGANE`, `APPAREIL`, `ENTITY`

**Sortie** : `output/rapport_organe_<slug>.html`

In [ ]:
# Paramètres papermill
ORGANE   = "Colon-Rectum-Anus"
APPAREIL = "APPAREIL DIGESTIF"
ENTITY   = "AP-HP"   # ou un GHU ex: "GHU Nord"

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from pathlib import Path
DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

import pandas as pd
aphp = pd.read_csv(DATA_DIR / 'aphp_data.csv')
reg  = pd.read_csv(DATA_DIR / 'regional_data.csv')
surv = pd.read_csv(DATA_DIR / 'survival_data.csv')

print(f"Organe    : {ORGANE}")
print(f"Appareil  : {APPAREIL}")
print(f"Entité    : {ENTITY}")

d = aphp[(aphp.entite == ENTITY) & (aphp.appareil == APPAREIL) & (aphp.organe == ORGANE)]
print(f"Lignes    : {len(d)}")
d

## Visualisations

In [ ]:
from chart_utils import (
    line_evolution, donut_market_share, TREATMENT_COLS, GHU_LIST,
    survival_by_stage, survival_evolution, delay_evolution, heatmap_organes
)

# Évolution patients
org_data = aphp[(aphp.appareil == APPAREIL) & (aphp.organe == ORGANE)]
fig = line_evolution(org_data, 'annee', 'nb_patients', 'entite',
                     f'Patients — {ORGANE}',
                     entities=[ENTITY] + (GHU_LIST if ENTITY == 'AP-HP' else ['AP-HP']))
fig.show()

In [ ]:
# Survie par stade
last_year = int(aphp.annee.max())
fig_s = survival_by_stage(surv, ENTITY, APPAREIL, organe=ORGANE, year=last_year)
fig_s.show()

In [ ]:
# Évolution de la survie
fig_se = survival_evolution(surv, ENTITY, APPAREIL, organe=ORGANE, stade='II')
fig_se.show()

In [ ]:
# Délais
fig_d = delay_evolution(aphp, ENTITY, APPAREIL, organe=ORGANE)
fig_d.show()

## Génération HTML

In [ ]:
from report_builder import build_rapport_organe

out = build_rapport_organe(ORGANE, APPAREIL, DATA_DIR, OUTPUT_DIR,
                            entity=ENTITY, aphp=aphp, reg=reg, surv=surv)
print(f"Rapport généré : {out}")